# M03 — Magnetic survey admission and spatial holdout

**Level 2 · Prerequisites: foundations notebook; read time: 25 minutes.**

## Objectives

Execute the strict product parser and geometry planner on the committed authored 288-row protocol fixture. It has no measured magnetic response. This is a partition/leakage-control example, not corrections, gridding, continuation or inversion.

## Setup

Use the already reviewed Python 3.12 pipeline environment. This notebook installs nothing.

Import repository code and confirm its source file.

In [ ]:
import runpy,sys
from pathlib import Path
root=Path.cwd().parents[2];sys.path.insert(0,str(root/'data-pipeline'))
from magnetic_survey_json import parse_request
from magnetic_survey import plan_geometry
support=runpy.run_path(str(root/'tests/data/magnetic_survey_support.py'))
# Verify the actual planner's source location.
print(Path(__import__('magnetic_survey').__file__).resolve())

## Data

Execute the strict product parser and geometry planner on the committed authored 288-row protocol fixture. It has no measured magnetic response. This is a partition/leakage-control example, not corrections, gridding, continuation or inversion.

**Files:** tests/data/magnetic_survey_support.py, tests/fixtures/magnetic_survey/generate.py, docs/methods/magnetic-survey/README.md

## Method

The contract fixes ENU metres, source/flight grouping, deterministic spatial blocks, an outer holdout and buffered folds. Rows sharing group or block stay together; rank and coverage are checked before likelihood materialization. This controls spatial leakage but does not correct diurnal variation, IGRF, altitude or provider grids.

The fixture provides east/north/up coordinates in metres, flight-group IDs, an explicit spatial block width and a 200 m buffer. The planner unions rows sharing either a flight group or a spatial block; transitive connections form indivisible units. It deterministically orders those units, holds out roughly one fifth for outer evaluation, then creates development folds. For each fold, fit rows within distance b of a validation or outer row are removed: retain i only if minⱼ∈H ‖xᵢ−xⱼ‖₂>b. This is a leakage-control rule, not a magnetic correction. Verde’s spatial train/test splitting is the related reference [API](https://www.fatiando.org/verde/latest/api/generated/verde.train_test_split.html).

The planner also checks geometric rank: for centered horizontal coordinates X=[1,e−ē,n−n̄], rank(X)=3 is required so the split is not collapsed to a line. This checks coordinate support, not field-property identifiability. The fixture has twelve flight lines and 288 positions but zero magnetic response. The figure shows exact row membership from fold 1, including buffered exclusions. Current eligibility explicitly withholds online/full-method acceptance because licensing, lineage, a validated likelihood and optimizer binding are unresolved.


Execute the product method on the declared control.

In [ ]:
raw=support['encode'](support['request']())
# Validate exact serialized bytes and compute the deterministic geometry seal.
handle=parse_request(raw);plan=plan_geometry(handle)
print({'schema':handle.metadata()['schema'],'partition_sha256':plan['partition']['sha256'],'outer_rows':len(plan['partition']['outer_rows']),'folds':len(plan['partition']['folds']),'inverse_run':False})

Blocked spatial splitting is preferable to random point splits for correlated flight lines. Verde documents spatial train/test splitting ([API reference](https://www.fatiando.org/verde/latest/api/generated/verde.train_test_split.html)); this product additionally seals group membership and buffers held-out rows. Current modules run only parser and geometry logic: no magnetic correction, kernel or optimizer.

Use the planner’s returned row indices directly. The chart is a map of the partition design, with physical coordinate axes in metres; its categories encode fit/holdout policy, never magnetic amplitude.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Plot exact product-planner indices on the fixture's authored local geometry; colors denote partition role only.
meta=handle.metadata(); xyz=np.asarray(meta['geometry']['receivers_m']['data'],dtype=float).reshape(-1,3)
outer=np.asarray(plan['partition']['outer_rows']['data'],dtype=int); f=plan['partition']['folds'][0]
fit=np.asarray(f['fit_rows']['data'],dtype=int); validation=np.asarray(f['validation_rows']['data'],dtype=int); buffered=np.asarray(f['buffered_rows']['data'],dtype=int)
dev=set(plan['partition']['development_rows']['data'])
assert not (set(outer)&dev) and not (set(fit)&set(validation) or set(fit)&set(buffered) or set(validation)&set(buffered))
fig,ax=plt.subplots(figsize=(9,5.4),constrained_layout=True)
for ids,label,color,marker in [(fit,'Fold 1 fit','#4477aa','o'),(validation,'Fold 1 validation','#ee7733','s'),(buffered,'Buffer excluded','#bbbbbb','x'),(outer,'Outer holdout','#aa3377','^')]:
 ax.scatter(xyz[ids,0],xyz[ids,1],label=f'{label} (n={len(ids)})',c=color,marker=marker,s=25)
ax.set(xlabel='Easting [m]',ylabel='Northing [m]',title='M03 geometry partition; no magnetic response')
ax.set_aspect('equal',adjustable='datalim'); ax.legend(ncol=2,fontsize=8)
print({'outer':len(outer),'fold_fit':len(fit),'fold_validation':len(validation),'buffered':len(buffered),**plan['claims']})
plt.show()


## Results

A successful run means schema and partition checks passed, not M03 processing, held-out prediction or field acceptance. On failure, retain the refusal; do not shrink buffers or synthesize measurements.

## Summary

The implemented slice is geometry admission. Full line/tie QC, corrections, gridding, source rights and actual held-out observations are separate requirements.




## References

- https://www.fatiando.org/verde/latest/api/generated/verde.train_test_split.html